# Multi-Domain Enterprise Assistant using Hybrid RAG

This notebook builds an enterprise question-answering assistant over HR, Finance, and IT policy documents. The main production-style pipeline is: document loading → chunking → embeddings → FAISS + BM25 hybrid retrieval → re-ranking → grounded answer generation.

**Note:** Cells before the multi-document section are earlier single-document/prototype experiments. The main project starts at the multi-document pipeline section.

In [ ]:
!pip install pymupdf

In [ ]:
import pymupdf

pdf_path = "/content/HR_Leave_Policy.pdf"

doc = pymupdf.open(pdf_path)

print("Number of pages:", len(doc))

Number of pages: 3


In [ ]:
full_text = ""

for page in doc:
    text = page.get_text()
    full_text += text + "\n"

print(full_text)

TechNova Solutions — HR Leave Policy 
TechNova Solutions 
HR Leave Policy 
Document ID: HR-LP-001 
Version: 1.0 
Effective Date: January 1, 2026 
Department: Human Resources 
 
1. Policy Overview 
TechNova Solutions provides eligible employees with different types of leave to support personal needs, 
health requirements, and planned time away from work. Employees must follow the leave application 
and approval procedures described in this policy. 
 
2. Casual Leave 
• Employees are eligible for 12 casual leave days per calendar year. 
• Casual leave may be used for personal matters, short-term requirements, or unforeseen 
situations that do not require medical leave. 
• Employees should submit a casual leave request through the company's leave management 
system whenever possible. 
• Casual leave should normally be requested at least one working day in advance. In emergency 
situations, the employee must inform their manager as soon as reasonably possible. 
 
3. Sick Leave 
• Employees

In [ ]:
import re

clean_text = re.sub(r'\n+', '\n', full_text)
clean_text = re.sub(r'[ \t]+', ' ', clean_text)

print(clean_text)

TechNova Solutions — HR Leave Policy 
TechNova Solutions 
HR Leave Policy 
Document ID: HR-LP-001 
Version: 1.0 
Effective Date: January 1, 2026 
Department: Human Resources 
 
1. Policy Overview 
TechNova Solutions provides eligible employees with different types of leave to support personal needs, 
health requirements, and planned time away from work. Employees must follow the leave application 
and approval procedures described in this policy. 
 
2. Casual Leave 
• Employees are eligible for 12 casual leave days per calendar year. 
• Casual leave may be used for personal matters, short-term requirements, or unforeseen 
situations that do not require medical leave. 
• Employees should submit a casual leave request through the company's leave management 
system whenever possible. 
• Casual leave should normally be requested at least one working day in advance. In emergency 
situations, the employee must inform their manager as soon as reasonably possible. 
 
3. Sick Leave 
• Employees

In [ ]:
chunk_size = 500
chunk_overlap = 50

chunks = []

start = 0

while start < len(clean_text):
    end = start + chunk_size
    chunk = clean_text[start:end]
    chunks.append(chunk)

    start = end - chunk_overlap

print("Number of chunks:", len(chunks))

Number of chunks: 9


In [ ]:
print("CHUNK 1")
print(chunks[0])

CHUNK 1
TechNova Solutions — HR Leave Policy 
TechNova Solutions 
HR Leave Policy 
Document ID: HR-LP-001 
Version: 1.0 
Effective Date: January 1, 2026 
Department: Human Resources 
 
1. Policy Overview 
TechNova Solutions provides eligible employees with different types of leave to support personal needs, 
health requirements, and planned time away from work. Employees must follow the leave application 
and approval procedures described in this policy. 
 
2. Casual Leave 
• Employees are eligible for 


In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_text(clean_text)

print("Number of chunks:", len(chunks))

Number of chunks: 9


In [ ]:
print("CHUNK 1")
print(chunks[0])

print("\n" + "="*80 + "\n")

print("CHUNK 2")
print(chunks[1])

CHUNK 1
TechNova Solutions — HR Leave Policy 
TechNova Solutions 
HR Leave Policy 
Document ID: HR-LP-001 
Version: 1.0 
Effective Date: January 1, 2026 
Department: Human Resources 
 
1. Policy Overview 
TechNova Solutions provides eligible employees with different types of leave to support personal needs, 
health requirements, and planned time away from work. Employees must follow the leave application 
and approval procedures described in this policy. 
 
2. Casual Leave


CHUNK 2
2. Casual Leave 
• Employees are eligible for 12 casual leave days per calendar year. 
• Casual leave may be used for personal matters, short-term requirements, or unforeseen 
situations that do not require medical leave. 
• Employees should submit a casual leave request through the company's leave management 
system whenever possible. 
• Casual leave should normally be requested at least one working day in advance. In emergency


In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [ ]:
first_chunk = chunks[0]

embedding = embedding_model.encode(first_chunk)

print("Vector created successfully!")
print("Vector dimension:", len(embedding))
print("First 10 values:", embedding[:10])

Vector created successfully!
Vector dimension: 384
First 10 values: [-0.03135155  0.06297081  0.02723997 -0.02592153  0.08311982  0.08404693
  0.03174293 -0.01983936 -0.10381768 -0.01022773]


In [ ]:
chunk_embeddings = embedding_model.encode(chunks)

print("Number of chunks:", len(chunk_embeddings))
print("Embedding dimension:", len(chunk_embeddings[0]))

Number of chunks: 9
Embedding dimension: 384


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.1 MB/s eta 0:00:00


In [ ]:
import faiss
import numpy as np

# Convert embeddings to float32
embedding_matrix = np.array(chunk_embeddings).astype("float32")

# Number of dimensions in each vector
dimension = embedding_matrix.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add all chunk embeddings
index.add(embedding_matrix)

print("FAISS index created successfully!")
print("Number of vectors stored:", index.ntotal)
print("Vector dimension:", dimension)

FAISS index created successfully!
Number of vectors stored: 9
Vector dimension: 384


In [ ]:
query = "How many casual leave days are employees eligible for?"

query_embedding = embedding_model.encode([query]).astype("float32")

print("Query embedding created!")
print("Query vector dimension:", query_embedding.shape[1])

Query embedding created!
Query vector dimension: 384


In [ ]:
k = 2

distances, indices = index.search(query_embedding, k)

print("Retrieved chunk indices:", indices)
print("Distances:", distances)

Retrieved chunk indices: [[1 3]]
Distances: [[0.41572046 0.7763163 ]]


In [ ]:
for i, index_id in enumerate(indices[0]):
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(chunks[index_id])


--- Retrieved Chunk 1 ---
2. Casual Leave 
• Employees are eligible for 12 casual leave days per calendar year. 
• Casual leave may be used for personal matters, short-term requirements, or unforeseen 
situations that do not require medical leave. 
• Employees should submit a casual leave request through the company's leave management 
system whenever possible. 
• Casual leave should normally be requested at least one working day in advance. In emergency

--- Retrieved Chunk 2 ---
4. Earned Leave 
• Employees are eligible for 15 earned leave days per calendar year. 
• Earned leave is intended for planned vacations, personal commitments, or extended periods 
away from work. 
 
• Employees should submit an earned leave request at least five working days before the planned 
leave date. 
 
5. Leave Approval Process 
• All planned leave requests must be submitted through the company's leave management system.


In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client configured successfully!")

Gemini client configured successfully!


In [ ]:
models = client.models.list()

for model in models:
    if "generateContent" in str(model.supported_actions):
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-p

In [ ]:
# Get the retrieved chunks
retrieved_chunks = [chunks[i] for i in indices[0]]

# Combine retrieved chunks
context = "\n\n".join(retrieved_chunks)

# User question
question = "How many casual leave days are employees eligible for?"

# RAG prompt
prompt = f"""
You are an AI enterprise assistant for TechNova Solutions.

Answer the user's question using ONLY the information provided
in the company document context below.

Do not use outside knowledge.
Do not make up information.

If the answer is not present in the context, say:
"I could not find this information in the provided company documents."

Company Document Context:
{context}

User Question:
{question}

Answer:
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

Employees are eligible for 12 casual leave days per calendar year.


In [ ]:
!pip install -q rank-bm25

In [ ]:
from rank_bm25 import BM25Okapi

# Convert each chunk into words
tokenized_chunks = [chunk.lower().split() for chunk in chunks]

# Create BM25 index
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created successfully!")
print("Number of documents indexed:", len(tokenized_chunks))

BM25 index created successfully!
Number of documents indexed: 9


In [ ]:
query = "How many casual leave days are employees eligible for?"

# Tokenize the query
tokenized_query = query.lower().split()

# Search using BM25
bm25_scores = bm25.get_scores(tokenized_query)

# Get top 2 chunk indices
bm25_top_indices = np.argsort(bm25_scores)[::-1][:2]

print("BM25 retrieved chunk indices:", bm25_top_indices)
print("BM25 scores:", bm25_scores[bm25_top_indices])

BM25 retrieved chunk indices: [1 3]
BM25 scores: [3.40919973 2.52417862]


In [ ]:
def min_max_normalize(scores):
    min_score = np.min(scores)
    max_score = np.max(scores)

    if max_score == min_score:
        return np.ones_like(scores)

    return (scores - min_score) / (max_score - min_score)


bm25_normalized = min_max_normalize(bm25_scores)

print("Normalized BM25 scores:")
print(bm25_normalized)

Normalized BM25 scores:
[0.51328533 1.         0.54038299 0.71447551 0.23773269 0.39733675
 0.20789401 0.         0.38487733]


In [ ]:
faiss_distances = distances[0]

# Convert distance into similarity
faiss_similarity = 1 / (1 + faiss_distances)

print("FAISS similarity scores:")
print(faiss_similarity)

FAISS similarity scores:
[0.70635414 0.56296283]


In [ ]:
# Search for all 9 chunks
all_distances, all_indices = index.search(
    query_embedding,
    len(chunks)
)

# Convert L2 distance to similarity
all_faiss_similarity = 1 / (1 + all_distances[0])

print("FAISS similarity for all chunks:")
print(all_faiss_similarity)

FAISS similarity for all chunks:
[0.70635414 0.56296283 0.56063324 0.501888   0.4582142  0.4375895
 0.42963114 0.4091735  0.37272117]


In [ ]:
# Combine BM25 and FAISS scores
hybrid_scores = (
    0.5 * bm25_normalized
    +
    0.5 * all_faiss_similarity
)

# Get top 3 chunks
hybrid_top_indices = np.argsort(hybrid_scores)[::-1][:3]

print("Hybrid scores:")
print(hybrid_scores)

print("\nTop 3 hybrid chunk indices:")
print(hybrid_top_indices)

print("\nTop 3 hybrid scores:")
print(hybrid_scores[hybrid_top_indices])

Hybrid scores:
[0.60981973 0.78148142 0.55050811 0.60818174 0.34797344 0.41746312
 0.31876257 0.20458674 0.37879925]

Top 3 hybrid chunk indices:
[1 0 3]

Top 3 hybrid scores:
[0.78148142 0.60981973 0.60818174]


In [ ]:
hybrid_retrieved_chunks = [
    chunks[i] for i in hybrid_top_indices
]

for rank, chunk in enumerate(hybrid_retrieved_chunks, start=1):
    print(f"\n--- Hybrid Retrieved Chunk {rank} ---")
    print(chunk)


--- Hybrid Retrieved Chunk 1 ---
2. Casual Leave 
• Employees are eligible for 12 casual leave days per calendar year. 
• Casual leave may be used for personal matters, short-term requirements, or unforeseen 
situations that do not require medical leave. 
• Employees should submit a casual leave request through the company's leave management 
system whenever possible. 
• Casual leave should normally be requested at least one working day in advance. In emergency

--- Hybrid Retrieved Chunk 2 ---
TechNova Solutions — HR Leave Policy 
TechNova Solutions 
HR Leave Policy 
Document ID: HR-LP-001 
Version: 1.0 
Effective Date: January 1, 2026 
Department: Human Resources 
 
1. Policy Overview 
TechNova Solutions provides eligible employees with different types of leave to support personal needs, 
health requirements, and planned time away from work. Employees must follow the leave application 
and approval procedures described in this policy. 
 
2. Casual Leave

--- Hybrid Retrieved Chunk 3

In [ ]:
# Combine the hybrid retrieved chunks into context
hybrid_context = "\n\n".join(hybrid_retrieved_chunks)

# User question
question = "How many casual leave days are employees eligible for?"

# Hybrid RAG prompt
hybrid_prompt = f"""
You are an AI enterprise assistant for TechNova Solutions.

Answer the user's question using ONLY the information
provided in the company document context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is not present in the context, say:
   "I could not find this information in the provided company documents."
4. Give a concise and direct answer.

Company Document Context:
{hybrid_context}

User Question:
{question}

Answer:
"""

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=hybrid_prompt
)

print("Final Answer:")
print(response.text)

Final Answer:
Employees are eligible for 12 casual leave days per calendar year.


In [ ]:
test_questions = [
    "How many casual leave days are employees eligible for?",
    "How many sick leave days are employees eligible for?",
    "How many earned leave days are employees eligible for?",
    "Can unused casual leave be carried forward to the following year?",
    "How many working days in advance should earned leave normally be requested?"
]

expected_answers = [
    "12 casual leave days",
    "10 sick leave days",
    "15 earned leave days",
    "No, unused casual leave cannot be carried forward",
    "5 working days"
]

print("Number of test questions:", len(test_questions))

Number of test questions: 5


In [ ]:
def hybrid_rag_answer(question, top_k=3):

    # 1. Create embedding for the question
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")

    # 2. BM25 search
    tokenized_query = question.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # 3. Normalize BM25 scores
    bm25_normalized = min_max_normalize(bm25_scores)

    # 4. FAISS search across all chunks
    all_distances, all_indices = index.search(
        query_embedding,
        len(chunks)
    )

    # 5. Convert FAISS distance to similarity
    faiss_similarity = 1 / (1 + all_distances[0])

    # 6. Hybrid score
    hybrid_scores = (
        0.5 * bm25_normalized
        +
        0.5 * faiss_similarity
    )

    # 7. Select top-k chunks
    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

    # 8. Get actual chunk text
    retrieved_chunks = [chunks[i] for i in top_indices]

    # 9. Create context
    context = "\n\n".join(retrieved_chunks)

    # 10. Generate answer using Gemini
    prompt = f"""
You are an AI enterprise assistant for TechNova Solutions.

Answer the user's question using ONLY the company document context.

Rules:
- Do not use outside knowledge.
- Do not invent information.
- If the answer is not present in the context, say:
"I could not find this information in the provided company documents."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, top_indices

In [ ]:
for i, question in enumerate(test_questions):

    answer, retrieved_indices = hybrid_rag_answer(question)

    print(f"\n{'='*60}")
    print(f"Question {i+1}: {question}")
    print(f"Retrieved chunks: {retrieved_indices}")
    print(f"Answer: {answer}")


Question 1: How many casual leave days are employees eligible for?
Retrieved chunks: [1 0 3]
Answer: Based on the provided company documents, employees are eligible for **12 casual leave days per calendar year**.

Question 2: How many sick leave days are employees eligible for?
Retrieved chunks: [2 3 1]
Answer: Employees are eligible for 10 sick leave days per calendar year.

Question 3: How many earned leave days are employees eligible for?
Retrieved chunks: [3 2 1]
Answer: Employees are eligible for 15 earned leave days per calendar year.

Question 4: Can unused casual leave be carried forward to the following year?
Retrieved chunks: [5 1 0]
Answer: No. Unused casual leave cannot be carried forward to the following calendar year.

Question 5: How many working days in advance should earned leave normally be requested?
Retrieved chunks: [1 3 2]
Answer: Earned leave should be requested at least five working days before the planned leave date.


In [ ]:
# Relevant chunk index for each question
relevant_chunk_indices = [1, 2, 3, 5, 3]

recall_results = []

for i, question in enumerate(test_questions):

    answer, retrieved_indices = hybrid_rag_answer(question, top_k=3)

    relevant_index = relevant_chunk_indices[i]

    if relevant_index in retrieved_indices:
        recall = 1
    else:
        recall = 0

    recall_results.append(recall)

    print(f"Question {i+1}: Recall@3 = {recall}")

# Overall Recall@3
recall_at_3 = sum(recall_results) / len(recall_results)

print("\nOverall Recall@3:", recall_at_3)
print("Recall@3 Percentage:", recall_at_3 * 100, "%")

Question 1: Recall@3 = 1
Question 2: Recall@3 = 1
Question 3: Recall@3 = 1
Question 4: Recall@3 = 1
Question 5: Recall@3 = 1

Overall Recall@3: 1.0
Recall@3 Percentage: 100.0 %


In [ ]:
precision_results = []

for i, question in enumerate(test_questions):

    answer, retrieved_indices = hybrid_rag_answer(question, top_k=3)

    relevant_index = relevant_chunk_indices[i]

    relevant_count = sum(
        1 for idx in retrieved_indices
        if idx == relevant_index
    )

    precision = relevant_count / 3
    precision_results.append(precision)

    print(f"Question {i+1}: Precision@3 = {precision:.2f}")

overall_precision = sum(precision_results) / len(precision_results)

print("\nOverall Precision@3:", round(overall_precision, 2))
print("Precision@3 Percentage:", round(overall_precision * 100, 2), "%")

Question 1: Precision@3 = 0.33
Question 2: Precision@3 = 0.33
Question 3: Precision@3 = 0.33
Question 4: Precision@3 = 0.33
Question 5: Precision@3 = 0.33

Overall Precision@3: 0.33
Precision@3 Percentage: 33.33 %


# 1. Multi-Document Enterprise Data

Here we introduce the additional Finance policy and prepare the three enterprise domains: Human Resources, Finance, and Information Technology.

In [ ]:
finance_text = """
TechNova Solutions — Employee Expense Reimbursement Policy

Document ID: FIN-ER-001
Version: 1.0
Department: Finance

1. Travel Reimbursement
Employees can claim reimbursement for approved business travel expenses.
Travel claims must be submitted within 15 calendar days after completing the business trip.

2. Meal Reimbursement
Employees may claim up to INR 500 per day for eligible business meal expenses.
Original receipts must be submitted with the reimbursement claim.

3. Approval Process
All reimbursement claims must be reviewed and approved by the employee's reporting manager.
Claims without required supporting documents may be rejected.

4. Policy Updates
The Finance Department may update the reimbursement policy when required.
Employees should refer to the latest approved version of the policy.
"""

it_text = """
TechNova Solutions — IT Security Policy

Document ID: IT-SP-001
Version: 1.0
Department: Information Technology

1. Password Security
Employees must use strong passwords for company accounts.
Passwords must not be shared with other employees.

2. Phishing Emails
Employees who receive a suspicious or phishing email should not click links,
open unknown attachments, or provide company credentials.
They should report the suspicious email to the IT Support team through the official
company communication channel.

3. Device Security
Employees must lock their company devices when leaving their workstation.
Company devices should not be used by unauthorized individuals.

4. Security Incidents
Employees must report suspected security incidents to the IT Support team as soon as possible.

5. Policy Updates
The Information Technology Department may update this policy when required.
Employees should refer to the latest approved version.
"""

print("Finance document loaded successfully!")
print("IT document loaded successfully!")

Finance document loaded successfully!
IT document loaded successfully!


# 2. Document Collection and Metadata

Each document is stored with a document ID, department, title, and text. This metadata lets the assistant identify where a retrieved chunk came from.

In [ ]:
documents = [
    {
        "document_id": "HR-LP-001",
        "department": "Human Resources",
        "title": "HR Leave Policy",
        "text": clean_text
    },
    {
        "document_id": "FIN-ER-001",
        "department": "Finance",
        "title": "Employee Expense Reimbursement Policy",
        "text": finance_text
    },
    {
        "document_id": "IT-SP-001",
        "department": "Information Technology",
        "title": "IT Security Policy",
        "text": it_text
    }
]

print("Total documents:", len(documents))

for doc in documents:
    print(
        f"{doc['document_id']} | "
        f"{doc['department']} | "
        f"{doc['title']}"
    )

Total documents: 3
HR-LP-001 | Human Resources | HR Leave Policy
FIN-ER-001 | Finance | Employee Expense Reimbursement Policy
IT-SP-001 | Information Technology | IT Security Policy


# 3. Document Chunking

Long policy documents are split into smaller overlapping chunks. Chunking makes retrieval more precise because the system can retrieve the specific section relevant to a question.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

all_chunks = []

for doc in documents:

    doc_chunks = splitter.split_text(doc["text"])

    for chunk in doc_chunks:
        all_chunks.append({
            "text": chunk,
            "document_id": doc["document_id"],
            "department": doc["department"],
            "title": doc["title"]
        })

print("Total chunks:", len(all_chunks))

for i, chunk in enumerate(all_chunks):
    print(
        f"\nCHUNK {i} | "
        f"{chunk['document_id']} | "
        f"{chunk['department']}"
    )
    print(chunk["text"][:250])

Total chunks: 10

CHUNK 0 | HR-LP-001 | Human Resources
TechNova Solutions — HR Leave Policy 
TechNova Solutions 
HR Leave Policy 
Document ID: HR-LP-001 
Version: 1.0 
Effective Date: January 1, 2026 
Department: Human Resources 
 
1. Policy Overview 
TechNova Solutions provides eligible employees with d

CHUNK 1 | HR-LP-001 | Human Resources
• Employees should submit a casual leave request through the company's leave management 
system whenever possible. 
• Casual leave should normally be requested at least one working day in advance. In emergency 
situations, the employee must inform th

CHUNK 2 | HR-LP-001 | Human Resources
4. Earned Leave 
• Employees are eligible for 15 earned leave days per calendar year. 
• Earned leave is intended for planned vacations, personal commitments, or extended periods 
away from work. 
 
• Employees should submit an earned leave request a

CHUNK 3 | HR-LP-001 | Human Resources
• Employees should confirm the approval status before taking planned leave

# 4. Embedding Generation

Each document chunk is converted into a numerical embedding using `all-MiniLM-L6-v2`. These vectors capture semantic meaning and are used for similarity search.

In [ ]:
# Extract only the text from our metadata-rich chunks
chunk_texts = [chunk["text"] for chunk in all_chunks]

# Create embeddings
all_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True
).astype("float32")

print("Embeddings created successfully!")
print("Number of embeddings:", len(all_embeddings))
print("Embedding dimension:", all_embeddings.shape[1])

Embeddings created successfully!
Number of embeddings: 10
Embedding dimension: 384


# 5. FAISS Semantic Index

FAISS stores the chunk embeddings and performs fast vector similarity search. This is the semantic retrieval component of the Hybrid RAG pipeline.

In [ ]:
import faiss
import numpy as np

# Create FAISS index
dimension = all_embeddings.shape[1]

multi_doc_index = faiss.IndexFlatL2(dimension)

# Add all 10 document chunk embeddings
multi_doc_index.add(all_embeddings)

print("Multi-document FAISS index created successfully!")
print("Number of vectors stored:", multi_doc_index.ntotal)
print("Vector dimension:", dimension)

Multi-document FAISS index created successfully!
Number of vectors stored: 10
Vector dimension: 384


# 6. BM25 Keyword Index

BM25 creates a keyword-based search index over the same chunks. It complements semantic retrieval by matching important words and phrases from the user query.

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize all document chunks
tokenized_chunks = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

# Create BM25 index for all 10 chunks
multi_doc_bm25 = BM25Okapi(tokenized_chunks)

print("Multi-document BM25 index created successfully!")
print("Number of documents indexed:", len(tokenized_chunks))

Multi-document BM25 index created successfully!
Number of documents indexed: 10


# 7. Finance Retrieval Test

This query demonstrates retrieval from the Finance policy. The goal is to verify that the relevant reimbursement chunk is retrieved.

In [ ]:
question = "What is the maximum meal reimbursement amount per day?"

# Query embedding
query_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
).astype("float32")

# FAISS retrieval
faiss_distances, faiss_indices = multi_doc_index.search(
    query_embedding,
    3
)

print("FAISS retrieved chunks:", faiss_indices[0])
print("FAISS distances:", faiss_distances[0])

# BM25 retrieval
tokenized_query = question.lower().split()
bm25_scores = multi_doc_bm25.get_scores(tokenized_query)

bm25_top_indices = np.argsort(bm25_scores)[::-1][:3]

print("\nBM25 retrieved chunks:", bm25_top_indices)
print("BM25 scores:", bm25_scores[bm25_top_indices])

# Display retrieved chunks
print("\n--- FAISS Retrieved Content ---")

for idx in faiss_indices[0]:
    print(f"\nChunk {idx}")
    print("Department:", all_chunks[idx]["department"])
    print("Document:", all_chunks[idx]["document_id"])
    print(all_chunks[idx]["text"][:500])

FAISS retrieved chunks: [6 7 1]
FAISS distances: [1.1964827 1.5430762 1.6636387]

BM25 retrieved chunks: [6 7 2]
BM25 scores: [6.02267816 2.52262821 1.9759283 ]

--- FAISS Retrieved Content ---

Chunk 6
Department: Finance
Document: FIN-ER-001
TechNova Solutions — Employee Expense Reimbursement Policy

Document ID: FIN-ER-001
Version: 1.0
Department: Finance

1. Travel Reimbursement
Employees can claim reimbursement for approved business travel expenses.
Travel claims must be submitted within 15 calendar days after completing the business trip.

2. Meal Reimbursement
Employees may claim up to INR 500 per day for eligible business meal expenses.
Original receipts must be submitted with the reimbursement claim.

3. Approval Process
All 

Chunk 7
Department: Finance
Document: FIN-ER-001
4. Policy Updates
The Finance Department may update the reimbursement policy when required.
Employees should refer to the latest approved version of the policy.

Chunk 1
Department: Human Resources
Documen

# 8. IT Security Retrieval Test

This query demonstrates retrieval from the IT Security policy, especially the phishing-email guidance.

In [ ]:
question = "What should employees do if they receive a suspicious phishing email?"

# Query embedding
query_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
).astype("float32")

# FAISS retrieval
faiss_distances, faiss_indices = multi_doc_index.search(
    query_embedding,
    3
)

print("FAISS retrieved chunks:", faiss_indices[0])
print("FAISS distances:", faiss_distances[0])

# BM25 retrieval
tokenized_query = question.lower().split()
bm25_scores = multi_doc_bm25.get_scores(tokenized_query)

bm25_top_indices = np.argsort(bm25_scores)[::-1][:3]

print("\nBM25 retrieved chunks:", bm25_top_indices)
print("BM25 scores:", bm25_scores[bm25_top_indices])

# Display retrieved content
print("\n--- Retrieved Content ---")

for idx in faiss_indices[0]:
    print(f"\nChunk {idx}")
    print("Department:", all_chunks[idx]["department"])
    print("Document:", all_chunks[idx]["document_id"])
    print(all_chunks[idx]["text"][:500])

FAISS retrieved chunks: [8 9 4]
FAISS distances: [0.95309377 1.2334416  1.6191261 ]

BM25 retrieved chunks: [8 4 0]
BM25 scores: [9.36384503 2.71265451 1.96058944]

--- Retrieved Content ---

Chunk 8
Department: Information Technology
Document: IT-SP-001
TechNova Solutions — IT Security Policy

Document ID: IT-SP-001
Version: 1.0
Department: Information Technology

1. Password Security
Employees must use strong passwords for company accounts.
Passwords must not be shared with other employees.

2. Phishing Emails
Employees who receive a suspicious or phishing email should not click links,
open unknown attachments, or provide company credentials.
They should report the suspicious email to the IT Support team through the official
company communicat

Chunk 9
Department: Information Technology
Document: IT-SP-001
4. Security Incidents
Employees must report suspected security incidents to the IT Support team as soon as possible.

5. Policy Updates
The Information Technology Department may up

# 9. Hybrid RAG Function

This function combines FAISS semantic similarity and BM25 keyword scores to produce a stronger set of candidate chunks for the user question.

In [ ]:
def multi_document_hybrid_rag(question, top_k=3):

    # 1. Query embedding
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")

    # 2. FAISS similarity
    faiss_distances, _ = multi_doc_index.search(
        query_embedding,
        len(all_chunks)
    )

    faiss_similarity = 1 / (1 + faiss_distances[0])

    # 3. BM25 scores
    tokenized_query = question.lower().split()
    bm25_scores = multi_doc_bm25.get_scores(tokenized_query)

    # 4. Normalize BM25
    bm25_normalized = min_max_normalize(bm25_scores)

    # 5. Hybrid score
    hybrid_scores = (
        0.5 * bm25_normalized
        + 0.5 * faiss_similarity
    )

    # 6. Top-k chunks
    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]

    # 7. Build context with metadata
    context_parts = []

    for idx in top_indices:
        chunk = all_chunks[idx]

        context_parts.append(
            f"""
Document ID: {chunk['document_id']}
Department: {chunk['department']}
Document Title: {chunk['title']}

Content:
{chunk['text']}
"""
        )

    context = "\n\n".join(context_parts)

    # 8. Gemini prompt
    prompt = f"""
You are an AI enterprise assistant for TechNova Solutions.

Answer the user's question using ONLY the provided company documents.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is not present in the documents, say:
"I could not find this information in the provided company documents."
4. Give a concise and accurate answer.
5. Mention the relevant department when useful.

Company Documents:
{context}

User Question:
{question}

Answer:
"""

    # 9. Generate answer
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, top_indices, hybrid_scores[top_indices]

# 10. Multi-Domain Hybrid RAG Demo

These test questions cover HR, Finance, and IT. The output shows which chunks were retrieved and the generated answer.

In [ ]:
multi_domain_questions = [
    "How many casual leave days are employees eligible for?",
    "What is the maximum meal reimbursement amount per day?",
    "What should employees do if they receive a suspicious phishing email?"
]

for i, question in enumerate(multi_domain_questions, start=1):

    answer, retrieved_indices, scores = multi_document_hybrid_rag(question)

    print(f"\n{'='*70}")
    print(f"Question {i}: {question}")
    print(f"Retrieved chunks: {retrieved_indices}")
    print(f"Hybrid scores: {scores}")
    print(f"\nAnswer: {answer}")


Question 1: How many casual leave days are employees eligible for?
Retrieved chunks: [0 1 2]
Hybrid scores: [0.83235294 0.74619668 0.55277044]

Answer: According to the Human Resources department's HR Leave Policy, employees are eligible for **12 casual leave days** per calendar year.

Question 2: What is the maximum meal reimbursement amount per day?
Retrieved chunks: [6 7 2]
Hybrid scores: [0.66554312 0.34485935 0.31733214]

Answer: According to the Finance Department's Employee Expense Reimbursement Policy (FIN-ER-001), the maximum meal reimbursement amount is INR 500 per day for eligible business meal expenses.

Question 3: What should employees do if they receive a suspicious phishing email?
Retrieved chunks: [8 0 4]
Hybrid scores: [0.67475334 0.3401486  0.31376116]

Answer: According to the Information Technology department's IT Security Policy, employees who receive a suspicious or phishing email should:

* **Not** click any links, open unknown attachments, or provide company c

# 11. Re-Ranker Setup

A cross-encoder re-ranker is loaded to score the relationship between each user question and its retrieved candidate chunk.

In [ ]:
!pip install -q sentence-transformers

# 12. Re-Ranker Model

The cross-encoder assigns relevance scores to question–chunk pairs. Higher relevance means the chunk is a better candidate for the final context.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Re-ranker model loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Re-ranker model loaded successfully!


# 13. Re-Ranking Function

This function takes the hybrid retrieval candidates and reorders them according to the cross-encoder relevance score.

In [ ]:
def rerank_chunks(question, candidate_indices):

    # Create question-chunk pairs
    pairs = [
        [question, all_chunks[idx]["text"]]
        for idx in candidate_indices
    ]

    # Get relevance scores from Cross-Encoder
    scores = reranker.predict(pairs)

    # Sort candidates by re-ranker score
    ranked = sorted(
        zip(candidate_indices, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked


# Test question
question = "What is the maximum meal reimbursement amount per day?"

# Get FAISS candidates
query_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
).astype("float32")

faiss_distances, faiss_indices = multi_doc_index.search(
    query_embedding,
    5
)

candidate_indices = faiss_indices[0]

# Re-rank them
reranked_results = rerank_chunks(
    question,
    candidate_indices
)

print("Original FAISS candidates:", candidate_indices)

print("\nRe-ranked results:")

for idx, score in reranked_results:
    print(f"\nChunk {idx} | Re-ranker Score: {score:.4f}")
    print("Department:", all_chunks[idx]["department"])
    print("Document:", all_chunks[idx]["document_id"])
    print(all_chunks[idx]["text"][:300])

Original FAISS candidates: [6 7 1 0 4]

Re-ranked results:

Chunk 6 | Re-ranker Score: 3.6856
Department: Finance
Document: FIN-ER-001
TechNova Solutions — Employee Expense Reimbursement Policy

Document ID: FIN-ER-001
Version: 1.0
Department: Finance

1. Travel Reimbursement
Employees can claim reimbursement for approved business travel expenses.
Travel claims must be submitted within 15 calendar days after completing the business

Chunk 7 | Re-ranker Score: -9.4538
Department: Finance
Document: FIN-ER-001
4. Policy Updates
The Finance Department may update the reimbursement policy when required.
Employees should refer to the latest approved version of the policy.

Chunk 1 | Re-ranker Score: -10.5898
Department: Human Resources
Document: HR-LP-001
• Employees should submit a casual leave request through the company's leave management 
system whenever possible. 
• Casual leave should normally be requested at least one working day in advance. In emergency 
situations, the employee must 

# 14. Final Hybrid RAG + Re-Ranking Pipeline

This is the main retrieval pipeline: query embedding → FAISS/BM25 hybrid retrieval → candidate selection → cross-encoder re-ranking → final relevant context.

In [ ]:
def reranked_hybrid_rag(question, candidate_k=5, final_k=3):

    # 1. Query embedding
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")

    # 2. FAISS retrieval
    faiss_distances, _ = multi_doc_index.search(
        query_embedding,
        len(all_chunks)
    )

    faiss_similarity = 1 / (1 + faiss_distances[0])

    # 3. BM25 retrieval
    tokenized_query = question.lower().split()
    bm25_scores = multi_doc_bm25.get_scores(tokenized_query)

    # 4. Normalize BM25
    bm25_normalized = min_max_normalize(bm25_scores)

    # 5. Hybrid scores
    hybrid_scores = (
        0.5 * bm25_normalized
        + 0.5 * faiss_similarity
    )

    # 6. Get candidate chunks
    candidate_indices = np.argsort(hybrid_scores)[::-1][:candidate_k]

    # 7. Re-rank candidates
    reranked = rerank_chunks(
        question,
        candidate_indices
    )

    # 8. Select final chunks
    final_indices = [
        idx for idx, score in reranked[:final_k]
    ]

    # 9. Build context with metadata
    context_parts = []

    for idx in final_indices:

        chunk = all_chunks[idx]

        context_parts.append(
            f"""
Document ID: {chunk['document_id']}
Department: {chunk['department']}
Document Title: {chunk['title']}

Content:
{chunk['text']}
"""
        )

    context = "\n\n".join(context_parts)

    # 10. Gemini prompt
    prompt = f"""
You are an AI enterprise assistant for TechNova Solutions.

Answer the user's question using ONLY the provided company documents.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is not present in the documents, say:
"I could not find this information in the provided company documents."
4. Give a concise and accurate answer.
5. Mention the relevant department when useful.

Company Documents:
{context}

User Question:
{question}

Answer:
"""

    # 11. Generate final answer
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, final_indices, reranked

# 15. Re-Ranker Verification

This cell demonstrates the re-ranked candidates for a Finance question and verifies that the relevant Finance chunk is prioritized.

In [ ]:
question = "What is the maximum meal reimbursement amount per day?"

query_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
).astype("float32")

# Get 5 hybrid candidates
faiss_distances, _ = multi_doc_index.search(
    query_embedding,
    len(all_chunks)
)

faiss_similarity = 1 / (1 + faiss_distances[0])

tokenized_query = question.lower().split()
bm25_scores = multi_doc_bm25.get_scores(tokenized_query)
bm25_normalized = min_max_normalize(bm25_scores)

hybrid_scores = (
    0.5 * bm25_normalized +
    0.5 * faiss_similarity
)

candidate_indices = np.argsort(hybrid_scores)[::-1][:5]

# Re-rank
reranked_results = rerank_chunks(
    question,
    candidate_indices
)

print("Hybrid candidates:", candidate_indices)

print("\nRe-ranked results:")

for idx, score in reranked_results:
    print(
        f"Chunk {idx} | "
        f"Score: {score:.4f} | "
        f"Department: {all_chunks[idx]['department']} | "
        f"Document: {all_chunks[idx]['document_id']}"
    )

Hybrid candidates: [6 7 2 1 3]

Re-ranked results:
Chunk 6 | Score: 3.6856 | Department: Finance | Document: FIN-ER-001
Chunk 7 | Score: -9.4538 | Department: Finance | Document: FIN-ER-001
Chunk 1 | Score: -10.5898 | Department: Human Resources | Document: HR-LP-001
Chunk 2 | Score: -10.8576 | Department: Human Resources | Document: HR-LP-001
Chunk 3 | Score: -11.2470 | Department: Human Resources | Document: HR-LP-001


# 16. LoRA Fine-Tuning Experiment

A small supervised dataset is prepared to explore LoRA adaptation for enterprise-style question answering. This is an experimental component, not the primary retrieval pipeline.

In [ ]:
lora_dataset = [
    {
        "instruction": "How many casual leave days are employees eligible for?",
        "context": "Employees are eligible for 12 casual leave days per calendar year.",
        "response": "Employees are eligible for 12 casual leave days per calendar year."
    },
    {
        "instruction": "How many sick leave days are employees eligible for?",
        "context": "Employees are eligible for 10 sick leave days per calendar year.",
        "response": "Employees are eligible for 10 sick leave days per calendar year."
    },
    {
        "instruction": "How many earned leave days are employees eligible for?",
        "context": "Employees are eligible for 15 earned leave days per calendar year.",
        "response": "Employees are eligible for 15 earned leave days per calendar year."
    },
    {
        "instruction": "Can unused casual leave be carried forward?",
        "context": "Unused casual leave cannot be carried forward to the following calendar year.",
        "response": "No. Unused casual leave cannot be carried forward to the following calendar year."
    },
    {
        "instruction": "What is the maximum meal reimbursement amount per day?",
        "context": "Employees may claim up to INR 500 per day for eligible business meal expenses.",
        "response": "The maximum eligible meal reimbursement is INR 500 per day."
    },
    {
        "instruction": "How long after a business trip can a travel claim be submitted?",
        "context": "Travel claims must be submitted within 15 calendar days after completing the business trip.",
        "response": "Travel claims must be submitted within 15 calendar days after completing the business trip."
    },
    {
        "instruction": "What should employees do if they receive a phishing email?",
        "context": "Employees should not click links, open unknown attachments, or provide company credentials. They should report the suspicious email to IT Support.",
        "response": "Employees should avoid clicking links, opening unknown attachments, or sharing credentials, and should report the suspicious email to IT Support."
    },
    {
        "instruction": "Should employees share their company passwords?",
        "context": "Passwords must not be shared with other employees.",
        "response": "No. Employees must not share their company passwords with other employees."
    }
]

print("LoRA training examples:", len(lora_dataset))

LoRA training examples: 8


In [ ]:
from datasets import Dataset

lora_dataset_hf = Dataset.from_list(lora_dataset)

print(lora_dataset_hf)
print("\nColumns:", lora_dataset_hf.column_names)
print("Number of examples:", len(lora_dataset_hf))

Dataset({
    features: ['instruction', 'context', 'response'],
    num_rows: 8
})

Columns: ['instruction', 'context', 'response']
Number of examples: 8


In [ ]:
def format_lora_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Context:
{example['context']}

### Response:
{example['response']}"""
    }

formatted_dataset = lora_dataset_hf.map(format_lora_prompt)

print("Formatted dataset successfully!")
print("\nFirst training example:\n")
print(formatted_dataset[0]["text"])

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Formatted dataset successfully!

First training example:

### Instruction:
How many casual leave days are employees eligible for?

### Context:
Employees are eligible for 12 casual leave days per calendar year.

### Response:
Employees are eligible for 12 casual leave days per calendar year.


In [ ]:
!pip install -q transformers peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Base model loaded successfully!")
print("Model:", model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct


In [ ]:
import transformers
import peft

print("Transformers version:", transformers.__version__)
print("PEFT version:", peft.__version__)

Transformers version: 5.13.1
PEFT version: 0.19.1


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Fresh base model loaded!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Fresh base model loaded!


In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

print("LoRA configuration created!")

LoRA configuration created!


In [ ]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.5 MB/s eta 0:00:00


In [ ]:
import torchao

print("torchao version:", torchao.__version__)

torchao version: 0.18.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

print("Fresh base model loaded!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Fresh base model loaded!


In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

print("LoRA configuration created!")

LoRA configuration created!


In [ ]:
trainable_params = 0
total_params = 0

for param in model.parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

trainable_percentage = 100 * trainable_params / total_params

print("Trainable parameters:", trainable_params)
print("Total parameters:", total_params)
print(f"Trainable percentage: {trainable_percentage:.4f}%")

Trainable parameters: 494032768
Total parameters: 494032768
Trainable percentage: 100.0000%


In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=False
)

print("Tokenization completed successfully!")
print("Number of training examples:", len(tokenized_dataset))
print("Available columns:", tokenized_dataset.column_names)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenization completed successfully!
Number of training examples: 8
Available columns: ['instruction', 'context', 'response', 'text', 'input_ids', 'attention_mask', 'labels']


In [ ]:
from datasets import Dataset

lora_dataset = [
    {
        "instruction": "How many casual leave days are employees eligible for?",
        "context": "Employees are eligible for 12 casual leave days per calendar year.",
        "response": "Employees are eligible for 12 casual leave days per calendar year."
    },
    {
        "instruction": "How many sick leave days are employees eligible for?",
        "context": "Employees are eligible for 10 sick leave days per calendar year.",
        "response": "Employees are eligible for 10 sick leave days per calendar year."
    },
    {
        "instruction": "How many earned leave days are employees eligible for?",
        "context": "Employees are eligible for 15 earned leave days per calendar year.",
        "response": "Employees are eligible for 15 earned leave days per calendar year."
    },
    {
        "instruction": "Can unused casual leave be carried forward?",
        "context": "Unused casual leave cannot be carried forward to the following calendar year.",
        "response": "No. Unused casual leave cannot be carried forward to the following calendar year."
    },
    {
        "instruction": "What is the maximum meal reimbursement amount per day?",
        "context": "Employees may claim up to INR 500 per day for eligible business meal expenses.",
        "response": "The maximum eligible meal reimbursement is INR 500 per day."
    },
    {
        "instruction": "How long after a business trip can a travel claim be submitted?",
        "context": "Travel claims must be submitted within 15 calendar days after completing the business trip.",
        "response": "Travel claims must be submitted within 15 calendar days after completing the business trip."
    },
    {
        "instruction": "What should employees do if they receive a phishing email?",
        "context": "Employees should not click links, open unknown attachments, or provide company credentials. They should report the suspicious email to IT Support.",
        "response": "Employees should avoid clicking links, opening unknown attachments, or sharing credentials, and should report the suspicious email to IT Support."
    },
    {
        "instruction": "Should employees share their company passwords?",
        "context": "Passwords must not be shared with other employees.",
        "response": "No. Employees must not share their company passwords with other employees."
    }
]

lora_dataset_hf = Dataset.from_list(lora_dataset)

print("Dataset recreated:", len(lora_dataset_hf))

Dataset recreated: 8


In [ ]:
def format_lora_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Context:
{example['context']}

### Response:
{example['response']}"""
    }

formatted_dataset = lora_dataset_hf.map(format_lora_prompt)

print("Formatted dataset recreated successfully!")
print(formatted_dataset[0]["text"])

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Formatted dataset recreated successfully!
### Instruction:
How many casual leave days are employees eligible for?

### Context:
Employees are eligible for 12 casual leave days per calendar year.

### Response:
Employees are eligible for 12 casual leave days per calendar year.


In [ ]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=False
)

print("Tokenization completed successfully!")
print("Number of training examples:", len(tokenized_dataset))
print("Columns:", tokenized_dataset.column_names)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenization completed successfully!
Number of training examples: 8
Columns: ['instruction', 'context', 'response', 'text', 'input_ids', 'attention_mask', 'labels']


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./enterprise_lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

print("LoRA Trainer configured successfully!")

LoRA Trainer configured successfully!


In [ ]:
print("Starting LoRA training...")

train_result = trainer.train()

print("\nLoRA training completed successfully!")
print("Training loss:", train_result.training_loss)

Starting LoRA training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,2.420722
2,2.397204
3,1.782295
4,1.303780
5,0.915763
6,0.457574


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


LoRA training completed successfully!
Training loss: 1.5462229549884796


In [ ]:
output_dir = "./enterprise_lora_final"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("LoRA adapter saved successfully!")
print("Saved to:", output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

LoRA adapter saved successfully!
Saved to: ./enterprise_lora_final


In [ ]:
import torch

test_question = "How many casual leave days are employees eligible for?"

prompt = f"""### Instruction:
{test_question}

### Context:
Employees are eligible for 12 casual leave days per calendar year.

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Fine-tuned model output:")
print(generated_text)

Fine-tuned model output:
### Instruction:
How many casual leave days are employees eligible for?

### Context:
Employees are eligible for 12 casual leave days per calendar year.

### Response:
Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for 12 casual leave days per calendar year. They are eligible for 12 casual leave days per calendar year. Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for 12 casual leave days


In [ ]:
test_questions = [
    (
        "How many sick leave days are employees eligible for?",
        "Employees are eligible for 10 sick leave days per calendar year."
    ),
    (
        "What is the maximum meal reimbursement amount per day?",
        "Employees may claim up to INR 500 per day for eligible business meal expenses."
    ),
    (
        "What should employees do if they receive a suspicious phishing email?",
        "Employees should not click links, open unknown attachments, or provide company credentials. They should report the suspicious email to IT Support."
    )
]

for question, context in test_questions:

    prompt = f"""### Instruction:
{question}

### Context:
{context}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    output = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 70)
    print("QUESTION:", question)
    print("MODEL OUTPUT:")
    print(output)

QUESTION: How many sick leave days are employees eligible for?
MODEL OUTPUT:
### Instruction:
How many sick leave days are employees eligible for?

### Context:
Employees are eligible for 10 sick leave days per calendar year.

### Response:
Employees are eligible for 10 sick leave days per calendar year. Employees are eligible for 10 sick leave days per calendar year. Employees are eligible for 10 sick leave days per calendar year. Employees are eligible for 10 sick leave days per calendar year. Employees are eligible for 10 sick leave days per calendar year. Employees are eligible for 10 sick leave days
QUESTION: What is the maximum meal reimbursement amount per day?
MODEL OUTPUT:
### Instruction:
What is the maximum meal reimbursement amount per day?

### Context:
Employees may claim up to INR 500 per day for eligible business meal expenses.

### Response:
Employees are eligible for INR 500 per day for eligible business meal expenses. They are eligible for INR 500 per day for eligibl

# 17. LoRA V2: Context-Grounded Training Experiment

The second LoRA dataset explicitly instructs the model to answer using only the supplied context. This experiment was used to study whether fine-tuning improves grounded responses.

In [87]:
from datasets import Dataset

lora_v2_data = [
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: How many casual leave days are employees eligible for?\nEmployees are eligible for 12 casual leave days per calendar year.",
        "response": "Employees are eligible for 12 casual leave days per calendar year."
    },
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: How many sick leave days are employees eligible for?\nEmployees are eligible for 10 sick leave days per calendar year.",
        "response": "Employees are eligible for 10 sick leave days per calendar year."
    },
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: How many earned leave days are employees eligible for?\nEmployees are eligible for 15 earned leave days per calendar year.",
        "response": "Employees are eligible for 15 earned leave days per calendar year."
    },
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: What is the maximum meal reimbursement amount per day?\nEmployees may claim up to INR 500 per day for eligible business meal expenses.",
        "response": "Employees may claim up to INR 500 per day for eligible business meal expenses."
    },
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: How long after a business trip can a travel claim be submitted?\nTravel claims must be submitted within 15 calendar days after completing the business trip.",
        "response": "Travel claims must be submitted within 15 calendar days after completing the business trip."
    },
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: What should employees do if they receive a suspicious phishing email?\nEmployees should not click links, open unknown attachments, or provide company credentials. They should report the suspicious email to IT Support.",
        "response": "Employees should not click links, open unknown attachments, or provide company credentials. They should report the suspicious email to IT Support."
    },
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: Should employees share their company passwords?\nPasswords must not be shared with other employees.",
        "response": "No. Employees must not share their company passwords with other employees."
    },
    {
        "instruction": "Answer the question using ONLY the provided context. Do not add information that is not present in the context.",
        "context": "Question: Can unused casual leave be carried forward?\nUnused casual leave cannot be carried forward to the following calendar year.",
        "response": "No. Unused casual leave cannot be carried forward to the following calendar year."
    }
]

lora_v2_dataset = Dataset.from_list(lora_v2_data)

print("LoRA V2 dataset created successfully!")
print("Number of examples:", len(lora_v2_dataset))

LoRA V2 dataset created successfully!
Number of examples: 8


In [88]:
def format_v2_prompt(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Context:
{example['context']}

### Response:
"""

    return {
        "prompt": prompt,
        "response": example["response"]
    }


v2_formatted = lora_v2_dataset.map(format_v2_prompt)

print("V2 prompts formatted successfully!")
print(v2_formatted[0]["prompt"])
print("Expected response:")
print(v2_formatted[0]["response"])

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

V2 prompts formatted successfully!
### Instruction:
Answer the question using ONLY the provided context. Do not add information that is not present in the context.

### Context:
Question: How many casual leave days are employees eligible for?
Employees are eligible for 12 casual leave days per calendar year.

### Response:

Expected response:
Employees are eligible for 12 casual leave days per calendar year.


In [89]:
def tokenize_v2(example):
    prompt_tokens = tokenizer(
        example["prompt"],
        add_special_tokens=True,
        truncation=True,
        max_length=512
    )

    response_tokens = tokenizer(
        example["response"],
        add_special_tokens=False,
        truncation=True,
        max_length=512
    )

    input_ids = prompt_tokens["input_ids"] + response_tokens["input_ids"]
    attention_mask = [1] * len(input_ids)

    # Ignore prompt tokens during loss calculation
    labels = [-100] * len(prompt_tokens["input_ids"]) + response_tokens["input_ids"]

    # Keep everything within max length
    input_ids = input_ids[:512]
    attention_mask = attention_mask[:512]
    labels = labels[:512]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


tokenized_v2 = v2_formatted.map(
    tokenize_v2,
    remove_columns=v2_formatted.column_names
)

print("V2 response-only tokenization completed!")
print("Number of examples:", len(tokenized_v2))
print("Columns:", tokenized_v2.column_names)

Parameter 'function'=<function tokenize_v2 at 0x7947c0c2c180> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/8 [00:00<?, ? examples/s]

V2 response-only tokenization completed!
Number of examples: 8
Columns: ['input_ids', 'attention_mask', 'labels']


In [91]:
from peft import LoraConfig, TaskType

lora_v2_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

model.add_adapter(
    lora_v2_config,
    adapter_name="enterprise_lora_v2"
)

model.set_adapter("enterprise_lora_v2")

print("LoRA V2 adapter added successfully!")

ValueError: Adapter with name enterprise_lora_v2 already exists. Please use a different name.

In [92]:
model.set_adapter("enterprise_lora_v2")

In [93]:
from transformers import TrainingArguments, Trainer

v2_training_args = TrainingArguments(
    output_dir="./enterprise_lora_v2",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)

v2_trainer = Trainer(
    model=model,
    args=v2_training_args,
    train_dataset=tokenized_v2
)

print("V2 LoRA Trainer configured successfully!")

V2 LoRA Trainer configured successfully!


In [ ]:
print("Starting LoRA V2 training...")

v2_train_result = v2_trainer.train()

print("\nLoRA V2 training completed successfully!")
print("V2 Training loss:", v2_train_result.training_loss)

Starting LoRA V2 training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.094589
2,0.240050
3,0.233679
4,0.070419
5,0.022745
6,0.248466


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


LoRA V2 training completed successfully!
V2 Training loss: 0.1516580873479446


# 18. Fast LoRA V2 Training

A shorter LoRA V2 training configuration is used for experimentation. The final project relies on Hybrid RAG + re-ranking for factual document retrieval.

In [ ]:
v2_training_args_fast = TrainingArguments(
    output_dir="./enterprise_lora_v2_fast",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)

v2_trainer_fast = Trainer(
    model=model,
    args=v2_training_args_fast,
    train_dataset=tokenized_v2
)

print("Fast V2 Trainer configured successfully!")

Fast V2 Trainer configured successfully!


In [ ]:
print("Starting fast LoRA V2 training...")

v2_fast_result = v2_trainer_fast.train()

print("Fast LoRA V2 training completed!")
print("Training loss:", v2_fast_result.training_loss)

Starting fast LoRA V2 training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.072496
2,0.176811


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fast LoRA V2 training completed!
Training loss: 0.12465311959385872


In [94]:
v2_output_dir = "./enterprise_lora_v2_final"

model.save_pretrained(v2_output_dir)
tokenizer.save_pretrained(v2_output_dir)

print("LoRA V2 adapter saved successfully!")
print("Saved to:", v2_output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

LoRA V2 adapter saved successfully!
Saved to: ./enterprise_lora_v2_final


# 19. LoRA V2 Evaluation

The trained adapter is tested against the enterprise questions to compare its generated responses with the source context.

In [95]:
import torch

v2_test_questions = [
    (
        "How many casual leave days are employees eligible for?",
        "Employees are eligible for 12 casual leave days per calendar year."
    ),
    (
        "How many sick leave days are employees eligible for?",
        "Employees are eligible for 10 sick leave days per calendar year."
    ),
    (
        "What is the maximum meal reimbursement amount per day?",
        "Employees may claim up to INR 500 per day for eligible business meal expenses."
    ),
    (
        "What should employees do if they receive a suspicious phishing email?",
        "Employees should not click links, open unknown attachments, or provide company credentials. They should report the suspicious email to IT Support."
    )
]

for question, context in v2_test_questions:

    prompt = f"""### Instruction:
Answer the question using ONLY the provided context. Do not add information that is not present in the context.

### Context:
Question: {question}
{context}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False
        )

    generated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 70)
    print("QUESTION:", question)
    print("V2 MODEL OUTPUT:")
    print(generated)

QUESTION: How many casual leave days are employees eligible for?
V2 MODEL OUTPUT:
### Instruction:
Answer the question using ONLY the provided context. Do not add information that is not present in the context.

### Context:
Question: How many casual leave days are employees eligible for?
Employees are eligible for 12 casual leave days per calendar year.

### Response:
Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for
QUESTION: How many sick leave days are employees eligible for?
V2 MODEL OUTPUT:
### Instruction:
Answer the question using ONLY the provided context. Do not add information that is not present in the context.

### Context:
Question: How many sick leave days are employees eligible for?
Employees are eligible for 10 sick leave days p

In [96]:
def clean_v2_response(generated_text):
    # Keep only text after ### Response:
    if "### Response:" in generated_text:
        answer = generated_text.split("### Response:", 1)[1]
    else:
        answer = generated_text

    # Stop if model starts generating another section
    for stop_text in ["### Instruction:", "### Context:"]:
        if stop_text in answer:
            answer = answer.split(stop_text, 1)[0]

    # Remove repeated sentences
    sentences = []
    seen = set()

    for sentence in answer.split("."):
        sentence = sentence.strip()

        if not sentence:
            continue

        key = sentence.lower()

        if key not in seen:
            seen.add(key)
            sentences.append(sentence)

    cleaned = ". ".join(sentences)

    if cleaned:
        cleaned += "."

    return cleaned.strip()

# 20. Response Cleaning / Evaluation Output

Generated responses are cleaned so that the final answer can be inspected separately from the prompt template.

In [97]:
print("Cleaned V2 Answers:")
print("=" * 70)

for question, context in v2_test_questions:

    prompt = f"""### Instruction:
Answer the question using ONLY the provided context. Do not add information that is not present in the context.

### Context:
Question: {question}
{context}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False
        )

    generated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    cleaned_answer = clean_v2_response(generated)

    print("Question:", question)
    print("Answer:", cleaned_answer)
    print()

Cleaned V2 Answers:
Question: How many casual leave days are employees eligible for?
Answer: Employees are eligible for 12 casual leave days per calendar year. Employees are eligible for.

Question: How many sick leave days are employees eligible for?
Answer: Employees are eligible for 10 sick leave days per calendar year. Employees are eligible for.

Question: What is the maximum meal reimbursement amount per day?
Answer: Employees are eligible for INR 500 per day for eligible business meal expenses. Context:
Employees are eligible for INR 500 per day for eligible business meal expenses. Response:
Employees are eligible for INR 500 per day for eligible business meal expenses. Context:.

Question: What should employees do if they receive a suspicious phishing email?
Answer: Employees must avoid clicking links, opening unknown attachments, or sharing credentials with other employees. They should report the suspicious email to IT Support. Employees must avoid clicking links,.



In [98]:
import re

def clean_v2_response(generated_text):
    # Take only what comes after the first Response marker
    if "### Response:" in generated_text:
        answer = generated_text.split("### Response:", 1)[1]
    else:
        answer = generated_text

    # Remove any newly generated sections
    answer = re.split(r"###\s*(Instruction|Context|Response):", answer)[0]

    # Remove repeated text by keeping only the first occurrence
    sentences = re.split(r"(?<=[.!?])\s+", answer.strip())

    unique_sentences = []
    seen = set()

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        # Ignore incomplete repeated fragments
        key = re.sub(r"\s+", " ", sentence.lower()).strip()

        if key not in seen:
            seen.add(key)
            unique_sentences.append(sentence)

    answer = " ".join(unique_sentences)

    # Remove trailing incomplete phrases
    incomplete_endings = [
        "Employees are eligible for",
        "Employees may claim up to",
        "Context:",
        "Response:",
        "Instruction:"
    ]

    for ending in incomplete_endings:
        if answer.endswith(ending):
            answer = answer[:-len(ending)].strip()

    return answer

In [100]:
import re
import torch

def clean_v2_response(generated_text):

    # Take only the answer after Response:
    if "### Response:" in generated_text:
        answer = generated_text.split("### Response:", 1)[1]
    else:
        answer = generated_text

    # Stop if model starts another section
    answer = re.split(
        r"###\s*(Instruction|Context|Response):",
        answer
    )[0]

    # Split into sentences
    sentences = re.split(r"(?<=[.!?])\s+", answer.strip())

    unique_sentences = []
    seen = set()

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        key = re.sub(r"\s+", " ", sentence.lower())

        if key not in seen:
            seen.add(key)
            unique_sentences.append(sentence)

    answer = " ".join(unique_sentences)

    return answer.strip()


def generate_v2_answer(question, context):

    prompt = f"""### Instruction:
Answer the question using ONLY the provided context. Do not add information that is not present in the context. Give one concise answer.

### Context:
Question: {question}
{context}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=35,
            do_sample=False,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return clean_v2_response(generated)


print("Testing cleaned V2 model...")
print("=" * 70)

for question, context in v2_test_questions:

    answer = generate_v2_answer(
        question,
        context
    )

    print("Question:", question)
    print("Answer:", answer)
    print()

Testing cleaned V2 model...
Question: How many casual leave days are employees eligible for?
Answer: Employees are eligiblefor 12 cazalveidays per calendar year. Employees are eligible for:

Question: How many sick leave days are employees eligible for?
Answer: Employees are eligiblefor 10sick leave days percalendar year. Employees are eligible for:

Question: What is the maximum meal reimbursement amount per day?
Answer: Employees are eligible for INR 1200 per eligible business meal expense. They must provide company credentials, a business meal reimbursement receipt and their company credentials. ### Response

Question: What should employees do if they receive a suspicious phishing email?
Answer: Employees must avoid clicking links, opening unknown attachments, and sharing credentials with other employees. Employees must report the suspiciousemail to IT Support. They should avoid clicking links. They must



# Final Project Summary

### Architecture
Documents → Chunking → 384-dimensional embeddings → FAISS + BM25 → Hybrid Retrieval → Cross-Encoder Re-Ranker → LLM Answer Generation

### Domains
- Human Resources
- Finance
- Information Technology

### Evaluation recorded during development
- Recall@3: 100%
- Precision@3: 33.33%

### Important note
LoRA fine-tuning was explored as an experiment. The Hybrid RAG + re-ranking pipeline is the main approach for factual enterprise document answering.